In [8]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import SAGEConv
import pickle
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
# ------------------------------------------------------------------------
# 1. ПОДГОТОВКА ДАННЫХ
# ------------------------------------------------------------------------

# [ИЗМЕНЕНО] Убрали emb_dim, добавили features_dict на вход
def df_to_homo_two_rel(df: pd.DataFrame, features_dict: dict, device="cpu"):
    # 1) глобальные id для узлов
    nodes = pd.concat([
        df[["type_entity_1","id_entity_1"]].rename(columns={"type_entity_1":"t","id_entity_1":"id"}),
        df[["type_entity_2","id_entity_2"]].rename(columns={"type_entity_2":"t","id_entity_2":"id"}),
    ], axis=0).drop_duplicates()

    key = list(zip(nodes["t"].astype(str), nodes["id"].astype(int)))
    node_map = {k:i for i,k in enumerate(key)}
    num_nodes = len(node_map)

    # 2) edge_index
    src = [node_map[(str(t), int(i))] for t,i in zip(df["type_entity_1"], df["id_entity_1"])]
    dst = [node_map[(str(t), int(i))] for t,i in zip(df["type_entity_2"], df["id_entity_2"])]
    edge_index = torch.tensor([src, dst], dtype=torch.long)

    # 3) edge_type (2 предиката -> 0/1)
    preds = df["predicate"].astype(str).unique().tolist()
    pred2id = {p:i for i,p in enumerate(sorted(preds))}
    edge_type = torch.tensor([pred2id[p] for p in df["predicate"].astype(str)], dtype=torch.long)

    edge_index, edge_type = to_undirected(edge_index, edge_type)

    # [ИЗМЕНЕНО] 4) Подготовка признаков разной размерности вместо рандома
    dims_dict = {}
    for t, emb_dict in features_dict.items():
        sample_emb = next(iter(emb_dict.values()))
        dims_dict[t] = len(sample_emb)
        
    unique_types = list(features_dict.keys())
    type2id = {t: i for i, t in enumerate(unique_types)}

    node_types_list = nodes["t"].astype(str).tolist()
    
    # [ДОБАВЛЕНО] Тензор, хранящий тип каждого узла (нужен для модели)
    node_type_tensor = torch.tensor([type2id[t] for t in node_types_list], dtype=torch.long)

    max_dim = max(dims_dict.values())
    x = torch.zeros(num_nodes, max_dim)

    node_ids_list = nodes["id"].astype(int).tolist()
    
    for i, (t, raw_id) in enumerate(zip(node_types_list, node_ids_list)):
        dim = dims_dict[t]
        emb = features_dict[t].get(raw_id, torch.zeros(dim))
        
        if not isinstance(emb, torch.Tensor):
            emb = torch.tensor(emb, dtype=torch.float)
            
        x[i, :dim] = emb

    data = Data(x=x, edge_index=edge_index)
    data.edge_type = edge_type  
    
    # [ДОБАВЛЕНО] Сохраняем типы узлов в объект графа
    data.node_type = node_type_tensor  

    return data, node_map, pred2id, type2id, dims_dict

In [5]:
class GraphSAGE(torch.nn.Module):
    # [ИЗМЕНЕНО] Инициализация теперь принимает словари размерностей и типов
    def __init__(self, dims_dict, type2id, hidden_dim):
        super().__init__()
        self.dims_dict = dims_dict
        self.type2id = type2id
        self.hidden_dim = hidden_dim

        # [ДОБАВЛЕНО] Модуль проекций для приведения векторов к единому размеру
        self.projections = nn.ModuleDict({
            str(t): nn.Linear(dim, hidden_dim) for t, dim in dims_dict.items()
        })

        self.conv1 = SAGEConv(hidden_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(0.1)

    # [ИЗМЕНЕНО] В forward добавлен аргумент node_type
    def forward(self, x_padded, edge_index, node_type):
        
        # [ДОБАВЛЕНО] Логика проекции
        x = torch.zeros(x_padded.size(0), self.hidden_dim, device=x_padded.device)

        for t, type_id in self.type2id.items():
            mask = (node_type == type_id)
            dim = self.dims_dict[t]
            # Вырезаем значимую часть вектора и проецируем
            x[mask] = self.projections[str(t)](x_padded[mask, :dim])

        # Дальше стандартные свертки
        x = self.conv1(x, edge_index)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.conv2(x, edge_index)

        return x

def decode(z, edge_label_index):
    src = z[edge_label_index[0]]
    dst = z[edge_label_index[1]]
    return (src * dst).sum(dim=1)

In [21]:
def train():
    model.train()
    optimizer.zero_grad()

    # [ИЗМЕНЕНО] Передаем node_type в модель
    z = model(train_data.x, train_data.edge_index, train_data.node_type)

    logits = decode(z, train_data.edge_label_index)
    loss_value = F.binary_cross_entropy_with_logits(logits, train_data.edge_label.float())

    loss_value.backward()
    optimizer.step()

    return loss_value.item()

@torch.no_grad()
def ranking_metrics(model, data_split, k_list=[1, 3, 10], N=1000):
    model.eval()
    
    # [ИЗМЕНЕНО] Передаем node_type в модель
    z = model(data_split.x, data_split.edge_index, data_split.node_type)

    edge_label_index = data_split.edge_label_index[:, :N]
    num_edges = edge_label_index.size(1)

    ranks = []

    for i in range(num_edges):
        src = edge_label_index[0, i]
        dst = edge_label_index[1, i]

        # score всех возможных dst для данного src
        scores = (z[src] * z).sum(dim=1)  # [num_nodes]
        _, sorted_idx = torch.sort(scores, descending=True)

        # позиция правильного ребра
        rank = (sorted_idx == dst).nonzero(as_tuple=True)[0].item() + 1  # 1-based
        ranks.append(rank)

    ranks = torch.tensor(ranks, dtype=torch.float)

    mrr = torch.mean(1.0 / ranks).item()

    hits = {}
    for k in k_list:
        hits_k = torch.mean((ranks <= k).float()).item()
        hits[k] = hits_k

    return mrr, hits

def load_and_merge_embeddings(folder_path):
    features_dict = {}
    
    # Список файлов из твоего скриншота
    # Ключи (слева) должны соответствовать значениям в колонках 'type_entity'
    mapping = {
        "DNA": "dnabert_DNA.pkl",
        "NucleicAmbigous": "dnabert_NucleicAmbigous.pkl",
        "NucleicMixed": "dnabert_NucleicMixed.pkl",
        "AA": "protein_esm_35M.pkl",
        "RNA": "rna_berta.pkl",
        "SmallMolecule": "sm_chemberta_10M_MTR.pkl"
    }
    
    for entity_type, file_name in mapping.items():
        file_path = os.path.join(folder_path, file_name)
        
        if os.path.exists(file_path):
            with open(file_path, 'rb') as f:
                # Загружаем словарь {raw_id: embedding}
                data = pickle.load(f)
                features_dict[entity_type] = data
                print(f"Loaded {entity_type}: {len(data)} entities")
        else:
            print(f"Warning: File {file_name} not found in {folder_path}")
            
    return features_dict

In [22]:
df = pd.read_csv('../data/edges/clean_triples.csv')

In [23]:
folder = "../data/dicts/bert_embeddings" 
all_features = load_and_merge_embeddings(folder)

Loaded DNA: 630 entities
Loaded NucleicAmbigous: 16 entities
Loaded NucleicMixed: 23 entities
Loaded AA: 71376 entities
Loaded RNA: 749 entities
Loaded SmallMolecule: 1301937 entities


In [ ]:
# Представим, что у нас уже есть df и features_dict
# [ИЗМЕНЕНО] Распаковываем новые возвращаемые значения: type2id и dims_dict

data, node_map, pre2id, type2id, dims_dict = df_to_homo_two_rel(df, all_features)

interacts_mask = data.edge_type == 0
similar_mask = data.edge_type == 1

edge_index_interacts = data.edge_index[:, interacts_mask]
edge_index_similarity = data.edge_index[:, similar_mask]

# создаем временный граф
# [ИЗМЕНЕНО] Обязательно передаем node_type в data_interacts, 
# чтобы RandomLinkSplit корректно его скопировал в train/val/test
data_interacts = Data(
    x=data.x,
    edge_index=edge_index_interacts,
    node_type=data.node_type 
)

# сплит ребер interacts
transform = RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    is_undirected=True,
    add_negative_train_samples=True,
    neg_sampling_ratio=1
)

train_data, val_data, test_data = transform(data_interacts)

# обратно добавляем has_similarity
for split_data in [train_data, val_data, test_data]:
    split_data.edge_index = torch.cat([split_data.edge_index, edge_index_similarity], dim=1)


In [29]:
train_data.node_type

tensor([4, 0, 0,  ..., 3, 3, 3], device='cuda:0')

In [26]:

# ------------------------------------------------------------------------
# 3. ОБУЧЕНИЕ И ВАЛИДАЦИЯ
# ------------------------------------------------------------------------

# Инициализируем модель с новыми аргументами
model = GraphSAGE(dims_dict=dims_dict, type2id=type2id, hidden_dim=16).to(device)

# Переносим данные на устройство (пример для train_data)
train_data = train_data.to(device)
val_data = val_data.to(device)
test_data = test_data.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

for epoch in range(1, 51):

    loss_value = train()

    if epoch % 10 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"Loss {loss_value:.4f} | "
        )



OutOfMemoryError: CUDA out of memory. Tried to allocate 3.07 GiB. GPU 0 has a total capacity of 7.61 GiB of which 687.88 MiB is free. Including non-PyTorch memory, this process has 6.53 GiB memory in use. Of the allocated memory 6.41 GiB is allocated by PyTorch, and 21.41 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)